# Where each engine wins: a scenario matrix

Three scenarios, same middleware. S1: separable batch (sharing
territory). S2: cut fragments, one-shot vs repeated (reuse territory).
S3: changing bindings (trajectory territory). Each ends with when to
use which engine.

In [1]:
import os

for key in ("QULACS_NUM_THREADS", "OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[key] = "1"

import math
import random
import sys
import time
from dataclasses import replace
from pathlib import Path
from statistics import median

ROOT = Path().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "benchmarks"))
sys.path.insert(0, str(ROOT / "tests"))

from lazyqc import CutOptions, Parameter, PlanningCache, Program, calibrate_batch, plan_all
from lazyqc.batch_planner import plan_batch
from lazyqc.fixtures import build_requests, load_fixture
from lazyqc.ir import Gate
from reference_numpy import expectation, statevector
from tensor_baseline import prepare as prepare_tensor


def timed(fn, repeats=5):
    values, stamps = None, []
    for _ in range(repeats):
        start = time.perf_counter_ns()
        values = fn()
        stamps.append((time.perf_counter_ns() - start) / 1e6)
    return values, median(stamps)

## S1 — Separable batch: 12 qubits, 24 requests

Six independent pairs. lazyqc grouped shares each pair preparation;
quimb factorizes the same structure.

In [2]:
pairs = 6
program = Program(2 * pairs)
for j in range(pairs):
    a, b = 2 * j, 2 * j + 1
    program = program.ry(a, (j + 1) * math.pi / (2 * pairs + 1)).cx(a, b)
batch = []
for j in range(pairs):
    a, b = 2 * j, 2 * j + 1
    batch += [
        program.expectation({a: "Z"}),
        program.expectation({b: "Z"}),
        program.expectation({a: "Z", b: "Z"}),
        program.expectation({a: "X", b: "X"}),
    ]
grouped, g_ms = timed(lambda: plan_all(batch, strategy="grouped").collect())
print(f"lazyqc grouped: {g_ms:.3f} ms")
tensor_values, q_ms = timed(lambda: prepare_tensor(batch).collect())
print(f"quimb tensor:   {q_ms:.3f} ms")
print("agreement:", max(abs(a - b) for a, b in zip(grouped.values, tensor_values)))

lazyqc grouped: 0.627 ms


quimb tensor:   47.216 ms
agreement: 9.992007221626409e-16


## S2 — Cut fragments: QAOA star, one-shot vs repeated

Four qubits, three requests, at most two CX cuts. Cold joint planning
loses (search charged once); warm cached sharing wins per collection.
quimb re-prepares every call. The crossover R decides.

In [3]:
fixture, cases = load_fixture(ROOT / "benchmarks/p4_p5.json")
star = next(c for c in cases if c["name"] == "qaoa_star")
width = star["fragment_width"]
model, _ = calibrate_batch(widths=(1, 2, 3, 4, 6, 8), repeats=5)
options = CutOptions(
    max_fragment_qubits=width, max_cuts=star["max_cuts"], **fixture["cut_limits"]
)
limits = dict(
    optimization="zx",
    strategy="auto",
    cost_model=model,
    workers=1,
    max_state_qubits=width,
    max_state_bytes=16 * 2**width,
    total_state_bytes=32 * 2**width,
)
star_requests = build_requests(star)

b3_cold, b3_cold_ms = timed(
    lambda: plan_all(star_requests, cutting=replace(options, search="independent"), **limits)
)
b5_cold, b5_cold_ms = timed(
    lambda: plan_all(star_requests, cutting=replace(options, search="joint"), **limits)
)
(_, tensor_ms) = timed(lambda: prepare_tensor(star_requests).collect())
print(f"one-shot cold: B3={b3_cold_ms:.1f} ms, B5={b5_cold_ms:.1f} ms, quimb={tensor_ms:.1f} ms")
print(f"tasks: B3={len(b3_cold.selected.tasks)}, B5={len(b5_cold.selected.tasks)}")

one-shot cold: B3=63.3 ms, B5=278.7 ms, quimb=33.4 ms
tasks: B3=76, B5=58


In [4]:
def warm_per_call(search, calls=10):
    cache = PlanningCache(8)
    stamps = []
    for _ in range(calls):
        start = time.perf_counter_ns()
        plan_all(star_requests, cutting=replace(options, search=search), cache=cache, **limits).collect()
        stamps.append((time.perf_counter_ns() - start) / 1e6)
    return median(stamps)


b3_warm = warm_per_call("independent")
b5_warm = warm_per_call("joint")
print(f"warm per collection: B3={b3_warm:.2f} ms, B5={b5_warm:.2f} ms, quimb={tensor_ms:.2f} ms")
saving = b3_warm - b5_warm
extra = b5_cold_ms - b3_cold_ms
import math as _math

crossover = _math.ceil(extra / saving) if saving > 0 else None
print(f"joint repays its extra search after ~{crossover} repeated collections")
print("verdict one-shot: quimb or B3; verdict repeated: cached joint")
print("caveat: warm edges under ~1 ms are noise-dominated (parity band)")


warm per collection: B3=2.75 ms, B5=2.29 ms, quimb=33.39 ms
joint repays its extra search after ~468 repeated collections
verdict one-shot: quimb or B3; verdict repeated: cached joint
caveat: warm edges under ~1 ms are noise-dominated (parity band)


## S3 — Changing bindings: structural replay

Same star topology, new angles per binding (a VQE step). The binding-keyed
cache misses every time; the structural template replays partitions and
cuts. v1 scope: `optimization="none"`, so topology is stable by
construction.

In [5]:
operations, counter = [], 0
for op in star["operations"]:
    if "angle_radians" in op:
        operations.append(Gate(op["gate"], tuple(op["qubits"]), Parameter(f"g{counter}")))
        counter += 1
    else:
        operations.append(Gate(op["gate"], tuple(op["qubits"])))
base = Program(star["qubits"], tuple(operations))
obs = [[(f["qubit"], f["pauli"]) for f in r["observable"]] for r in star["requests"]]
rng = random.Random(11)
bindings = [{f"g{i}": rng.uniform(-1, 1) for i in range(counter)} for _ in range(3)]
programs = [base.bind(**b) for b in bindings]

plain = dict(strategy="auto", optimization="none", workers=1, max_state_qubits=width,
               max_state_bytes=16 * 2**width, total_state_bytes=32 * 2**width)
seed_requests = [programs[0].expectation(dict(o)) for o in obs]
seed = plan_batch(seed_requests, cutting=replace(options, search="joint"), **plain)
template = seed.explain()["search"]["structural_template"]
print(f"seed: {len(seed.selected.tasks)} tasks, {seed.planning_ns/1e6:.0f} ms")

for step in (1, 2):
    req = [programs[step].expectation(dict(o)) for o in obs]
    _, full_ms = timed(lambda: plan_batch(req, cutting=replace(options, search="joint"), **plain), repeats=3)
    replay, hint_ms = timed(
        lambda: plan_batch(req, cutting=replace(options, search="joint"), _structural_hints=template, **plain),
        repeats=3,
    )
    full_check = plan_batch(req, cutting=replace(options, search="joint"), **plain)
    fidel = (len(replay.selected.tasks) == len(full_check.selected.tasks)
             and replay.selected.score == full_check.selected.score)
    state = statevector(4, [(g.name, g.qubits, g.angle) for g in programs[step].resolved_operations()])
    reference = tuple(expectation(state, r.observable.factors) for r in req)
    error = max(abs(a - b) for a, b in zip(replay.collect().values, reference))
    print(f"binding {step}: full={full_ms:.0f} ms, replay={hint_ms:.1f} ms, fidelity={fidel}, err={error:.1e}")


seed: 67 tasks, 96 ms


binding 1: full=170 ms, replay=5.1 ms, fidelity=True, err=1.9e-16


binding 2: full=93 ms, replay=4.5 ms, fidelity=True, err=3.3e-16


## Guidance: use which engine when

| Situation | Engine | Why |
|---|---|---|
| Small dense circuit, one batch | Direct Qulacs | No planning overhead; one state serves all |
| Separable/overlapping batch, one batch | lazyqc grouped (or quimb) | Shared preparations; payload capped |
| Cut fragments, one-shot | quimb or B3 | Cold joint search never repays once |
| Cut fragments, repeated batch | Cached joint (B5) | Repays after ~tens of collections |
| VQE angle updates | Structural replay | Topology cached, numbers rebound |
| Wide connected batch (12q+) | Dense full state | Tensor path search explodes; cones percolate |

No engine dominates: the crossover points (qubit count, reuse count,
contraction width) decide, and they are workload-specific.